# Comparação de Experimentos: MOEA + DVL vs Apenas MOEA

In [ ]:
import numpy as np
import random
import time
import pandas as pd

# Importando os Problemas, Algoritmos e Indicador de Qualidade
from src.problems.DTLZ import DTLZ1, DTLZ2, DTLZ3, DTLZ4
from src.MOEAs.NSGAIII import NSGAIII

from src.dvl.DVL import DVLFramework
from src.dvl.models.Linear import LinearModel
from src.QualityIndicator import HV

from src.MOEAs.mutations.PolynomialMutation import PolynomialMutation
from src.MOEAs.crossovers.SBXCrossover import SBXCrossover
from src.BinaryTournament import BinaryTournament
from src.MOEAs.sparsities.CrowdingDistance import CrowdingDistance

# Pontos de referência para o cálculo do Hypervolume
HV_REFERENCE_POINTS = {
    ("DTLZ1", 3): np.array([1.0, 1.0, 1.0], dtype=float),
    ("DTLZ1", 10): np.array([5.0] * 10, dtype=float),
    ("DTLZ2", 3): np.array([2.0, 2.0, 2.0], dtype=float),
    ("DTLZ2", 10): np.array([2.0] * 10, dtype=float),
    ("DTLZ3", 3): np.array([2.0, 2.0, 2.0], dtype=float),
    ("DTLZ3", 10): np.array([2.0] * 10, dtype=float),
    ("DTLZ4", 3): np.array([2.0, 2.0, 2.0], dtype=float),
    ("DTLZ4", 10): np.array([2.0] * 10, dtype=float),
}

In [11]:
def run_with_dvl(
    problem_class,      # Classe do problema
    moea_class,         # Classe do algoritmo evolucionário
    model_class,        # Classe do modelo de aprendizado de máquina
    m,                  # Número de objetivos do problema (M)
    k,                  # Parâmetro k do problema DTLZ
    max_evaluations,    # Número máximo de avaliações
    sample_size,        # Tamanho da amostra inicial (LHS) para treinar o DVL
    seed=42,
    run_moea=True
):
    np.random.seed(seed)
    random.seed(seed)
    
    problem = problem_class(numberOfObjectives=m, k=k)
    model = model_class()
    
    div_dict = {2: 99, 3: 12, 10: 3}
    num_div = div_dict.get(m, 12)
    
    framework = DVLFramework(
        pop_size=sample_size,
        max_eval=max_evaluations,
        ClassMoea=moea_class,
        model=model,
        problem=problem,
        reference_point_divisions=num_div,
        sampling_seed=seed,
        objective_transform="direction",
        run_moea=run_moea
    )
    
    population = framework.execute()
    return population

In [12]:
def run_without_dvl(
    problem_class,  # Classe do problema (ex: DTLZ1, DTLZ2)
    moea_class,     # Classe do algoritmo evolucionário (ex: NSGAIII, NSGAII)
    m,              # Número de objetivos do problema (M)
    k,              # Parâmetro k do problema DTLZ
    max_evaluations,# Número máximo de avaliações (max evaluations)
    seed=42
):
    import inspect
    from src.Util import ReferencePoint

    np.random.seed(seed)
    random.seed(seed)

    problem = problem_class(numberOfObjectives=m, k=k)
    
    div_dict = {2: 99, 3: 12, 10: 3}
    num_div = div_dict.get(m, 12)
    
    crossover = SBXCrossover(20.0, 0.9)
    mutation_probability = 1.0 / problem.numberOfDecisionVariables
    mutation = PolynomialMutation(mutation_probability, 20.0)
    selection = BinaryTournament()
    sparsity = CrowdingDistance()
    
    # Gera pontos de referência para deduzir tamanho de população se o algoritmo precisar
    ref_points = ReferencePoint().generateReferencePoints(m, num_div)
    pop_size = len(ref_points)

    # Identificar assinatura do construtor dinamicamente
    sig = inspect.signature(moea_class)
    params = sig.parameters
    
    kwargs = {}
    if "problem" in params:
        kwargs["problem"] = problem
    if "maxEvaluations" in params:
        kwargs["maxEvaluations"] = max_evaluations
    if "crossover" in params:
        kwargs["crossover"] = crossover
    if "mutation" in params:
        kwargs["mutation"] = mutation
    if "selection" in params:
        kwargs["selection"] = selection
    if "sparsity" in params:
        kwargs["sparsity"] = sparsity
    if "numberOfDivisions" in params:
        kwargs["numberOfDivisions"] = num_div
    if "populationSize" in params:
        kwargs["populationSize"] = pop_size
    if "offSpringPopulationSize" in params:
        kwargs["offSpringPopulationSize"] = pop_size if pop_size % 2 == 0 else pop_size + 1
        
    moea = moea_class(**kwargs)
    population = moea.execute()
    
    return population

In [13]:
def build_experiment_configs():
    configs = []
    
    # Parâmetros para 3 objetivos (k=10)
    m3_configs = {
        "DTLZ1": {"e": [250, 500, 1000, 1500, 10000], "s": [159, 227, 250, 300, 300]},
        "DTLZ2": {"e": [250, 500, 1000, 1500, 10000], "s": [159, 227, 600, 600, 600]},
        "DTLZ3": {"e": [250, 500, 1000, 1500, 10000], "s": [159, 227, 300, 300, 300]},
        "DTLZ4": {"e": [250, 500, 1000, 1500, 10000], "s": [159, 410, 410, 410, 500]}
    }
    
    # Parâmetros para 10 objetivos (k=1)
    m10_configs = {
        "DTLZ1": {"e": [250, 500, 1000, 1500, 10000], "s": [50, 112, 200, 200, 300]},
        "DTLZ2": {"e": [250, 500, 1000, 1500, 10000], "s": [50, 280, 300, 300, 300]},
        "DTLZ3": {"e": [250, 500, 1000, 1500, 10000], "s": [50, 112, 200, 200, 300]},
        "DTLZ4": {"e": [250, 500, 1000, 1500, 10000], "s": [50, 280, 300, 300, 300]}
    }
    
    # Gerando para m=3 (problem_k=10)
    for problem_name, data in m3_configs.items():
        for e, s in zip(data["e"], data["s"]):
            configs.append({
                "problem_name": problem_name,
                "m": 3,
                "problem_k": 10,
                "max_evaluations": e,
                "sample_size": s,
                "label": f"{problem_name}_m3_e{e}_s{s}"
            })
            
    # Gerando para m=10 (problem_k=1)
    for problem_name, data in m10_configs.items():
        for e, s in zip(data["e"], data["s"]):
            configs.append({
                "problem_name": problem_name,
                "m": 10,
                "problem_k": 1,
                "max_evaluations": e,
                "sample_size": s,
                "label": f"{problem_name}_m10_e{e}_s{s}"
            })
            
    return configs

In [14]:
def run_experiment(config, problem_class, algo_class, mode, seed):
    """Executa um único experimento, lidando com erros e retornando as métricas."""
    label = config["label"]
    problem_name = config["problem_name"]
    m = config["m"]
    k = config["problem_k"]
    max_evals = config["max_evaluations"]
    sample_size = config["sample_size"]
    
    result = {
        "label": label,
        "problem": problem_name,
        "algorithm": algo_class.__name__,
        "mode": mode,
        "m": m,
        "problem_k": k,
        "max_evaluations": max_evals,
        "sample_size": sample_size,
        "seed": seed,
        "cpu_time_seconds": 0.0,
        "population_size_final": None,
        "hypervolume": 0.0,
        "status": "success",
        "error_message": ""
    }
    
    start_cpu = time.process_time()
    try:
        if mode == "pure_moea":
            population = run_without_dvl(
                problem_class=problem_class,
                moea_class=algo_class,
                m=m,
                k=k,
                max_evaluations=max_evals,
                seed=seed
            )
        elif mode == "dvl_framework":
            population = run_with_dvl(
                problem_class=problem_class,
                moea_class=algo_class,
                model_class=LinearModel,
                m=m,
                k=k,
                max_evaluations=max_evals,
                sample_size=sample_size,
                seed=seed,
                run_moea=True
            )
        else:
            raise ValueError(f"Mode desconhecido: {mode}")
            
        end_cpu = time.process_time()
        result["cpu_time_seconds"] = end_cpu - start_cpu
        result["population_size_final"] = len(population)
        
        # Cálculo do Hypervolume Normalizado
        ref_point = HV_REFERENCE_POINTS.get((problem_name, m))
        if ref_point is not None:
            objectives_list = [sol.objectives for sol in population]
            indicator = HV(referencePoint=ref_point)
            hv_val = indicator.calculate(objectives_list)
            result["hypervolume"] = hv_val / np.prod(ref_point)
            
    except Exception as e:
        end_cpu = time.process_time()
        result["cpu_time_seconds"] = end_cpu - start_cpu
        result["status"] = "error"
        result["error_message"] = str(e)
        result["hypervolume"] = np.nan
        
    return result

In [ ]:
# Configurando o loop principal de experimentos
configs = build_experiment_configs()

problem_classes = {
    "DTLZ1": DTLZ1
    # "DTLZ2": DTLZ2,
    # "DTLZ3": DTLZ3,
    # "DTLZ4": DTLZ4
}
algorithms = [NSGAIII]

# Configurações de Execução
SEED = 42
NUM_RUNS = 20
results = []

for config in configs:
    problem_class = problem_classes.get(config["problem_name"])
    if not problem_class:
        continue
        
    for algo_class in algorithms:
        for _ in range(NUM_RUNS):
            # 1. MOEA Puro
            res_pure = run_experiment(config, problem_class, algo_class, mode="pure_moea", seed=SEED)
            results.append(res_pure)
            
            # 2. DVL + MOEA
            res_dvl = run_experiment(config, problem_class, algo_class, mode="dvl_framework", seed=SEED)
            results.append(res_dvl)

Executando NSGAIII puro para DTLZ1_m3_e250_s159 (seed=42)...
Evaluations: 91 de 250...
Evaluations: 187 de 250...
Executando DVL + NSGAIII para DTLZ1_m3_e250_s159 (seed=42)...
Executando NSGAIII puro para DTLZ1_m3_e250_s159 (seed=42)...
Evaluations: 91 de 250...
Evaluations: 187 de 250...
Executando DVL + NSGAIII para DTLZ1_m3_e250_s159 (seed=42)...
Executando NSGAIII puro para DTLZ1_m3_e250_s159 (seed=42)...
Evaluations: 91 de 250...
Evaluations: 187 de 250...
Executando DVL + NSGAIII para DTLZ1_m3_e250_s159 (seed=42)...
Executando NSGAIII puro para DTLZ1_m3_e250_s159 (seed=42)...
Evaluations: 91 de 250...
Evaluations: 187 de 250...
Executando DVL + NSGAIII para DTLZ1_m3_e250_s159 (seed=42)...
Executando NSGAIII puro para DTLZ1_m3_e250_s159 (seed=42)...
Evaluations: 91 de 250...
Evaluations: 187 de 250...
Executando DVL + NSGAIII para DTLZ1_m3_e250_s159 (seed=42)...
Executando NSGAIII puro para DTLZ1_m3_e250_s159 (seed=42)...
Evaluations: 91 de 250...
Evaluations: 187 de 250...
Execut

KeyboardInterrupt: 

In [ ]:
# Visualização dos Resultados das Execuções Raw
df = pd.DataFrame(results)
df

,label,problem,algorithm,mode,m,problem_k,max_evaluations,sample_size,seed,cpu_time_seconds,population_size_final,hypervolume,status,error_message
0,DTLZ1_m3_e250_s159,DTLZ1,NSGAIII,pure_moea,3,10,250,159,42,0.124209,88,0.162061,success,
1,DTLZ1_m3_e250_s159,DTLZ1,NSGAIII,dvl_framework,3,10,250,159,42,0.034268,91,0.383151,success,
2,DTLZ1_m3_e250_s159,DTLZ1,NSGAIII,pure_moea,3,10,250,159,43,0.111109,80,0.035251,success,
3,DTLZ1_m3_e250_s159,DTLZ1,NSGAIII,dvl_framework,3,10,250,159,43,0.034576,91,0.539809,success,
4,DTLZ1_m3_e250_s159,DTLZ1,NSGAIII,pure_moea,3,10,250,159,44,0.105969,91,0.163561,success,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,DTLZ1_m3_e500_s227,DTLZ1,NSGAIII,dvl_framework,3,10,500,227,59,0.180655,87,0.509695,success,
76,DTLZ1_m3_e500_s227,DTLZ1,NSGAIII,pure_moea,3,10,500,227,60,0.268139,69,0.578272,success,
77,DTLZ1_m3_e500_s227,DTLZ1,NSGAIII,dvl_framework,3,10,500,227,60,0.143313,87,0.436815,success,
78,DTLZ1_m3_e500_s227,DTLZ1,NSGAIII,pure_moea,3,10,500,227,61,0.244074,88,0.572831,success,


In [ ]:
# Agregação Estatística (Média e Desvio Padrão do HV)
if not df.empty and "hypervolume" in df.columns:
    summary_df = df.groupby(["problem", "m", "max_evaluations", "mode"]).agg(
        hv_mean=("hypervolume", "mean"),
        hv_std=("hypervolume", "std"),
        pop_size_mean=("population_size_final", "mean"),
        cpu_time_mean=("cpu_time_seconds", "mean"),
        successful_runs=("status", lambda x: (x == "success").sum())
    ).reset_index()
    display(summary_df)
else:
    print("Nenhum resultado disponível para resumir.")

,problem,m,max_evaluations,mode,hv_mean,hv_std,pop_size_mean,cpu_time_mean,successful_runs
0,DTLZ1,3,250,dvl_framework,0.262087,0.220299,91.00,0.036408,20
1,DTLZ1,3,250,pure_moea,0.400336,0.188689,85.45,0.107492,20
2,DTLZ1,3,500,dvl_framework,0.586405,0.168872,84.10,0.146781,20
3,DTLZ1,3,500,pure_moea,0.629727,0.149056,83.95,0.255906,20
